### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model
model1=init_chat_model("groq:openai/gpt-oss-20b")
response = model1.invoke("Why do parrots talk?")
response.content

'**Short answer:**  \nParrots talk because they are *vocal learners*—they have the brain circuitry and the physical tools to imitate sounds.  In the wild they use this skill to communicate with flock mates, to signal danger, to attract mates, and to build social bonds.  In a human‑domesticated setting, that same skill is redirected toward our voices, so they “talk” to us.\n\n---\n\n## 1. The biology of parrot “speech”\n\n| Feature | Why it matters |\n|---------|----------------|\n| **Syrinx (bird “voice box”)** | Located at the base of the trachea, it has many muscles and can produce a wide range of sounds—more flexible than the human larynx. |\n| **Large vocal learning brain** | Parrots have a well‑developed “vocal learning” area in the forebrain (the nucleus mesencephalicus lateralis pars dorsalis, or NMLD).  This region is analogous to the human Broca’s area and allows them to map sounds to motor commands. |\n| **High brain‑to‑body ratio** | Parrots are among the most intelligent bi

In [3]:
from langchain.tools import tool 

@tool
def get_weather(location: str) -> str:
    """Get the weather at a given location"""
    return f"The weather in {location} is sunny"

In [4]:
model_bind_tools = model1.bind_tools([get_weather])

In [6]:
response = model_bind_tools.invoke("What is the weather in New York?")
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function get_weather with location "New York".', 'tool_calls': [{'id': 'fc_b174cab2-be6f-45bb-a9f7-02da2588238d', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 129, 'total_tokens': 168, 'completion_time': 0.04920341, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.011528701, 'prompt_tokens_details': None, 'queue_time': 0.341534044, 'total_time': 0.060732111}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_1074f9ce08', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fcc3-d3bb-7d91-9df2-47c0f2c96b63-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_b174cab2-be6f-45bb-a9f7-02da2588238d', 'type': 'tool_call'}], invalid_tool_calls=[], us

In [7]:
for tool_call in response.tool_calls:
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

Tool:get_weather
Args:{'location': 'New York'}


### Tool Execution Loops

In [9]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_bind_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_bind_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


In [10]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What\'s the weather in Boston?" We need to call get_weather function.', 'tool_calls': [{'id': 'fc_7f5b7cb5-9266-43b5-a91b-b6b6cc7b99b2', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 127, 'total_tokens': 169, 'completion_time': 0.04698097, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.006237754, 'prompt_tokens_details': None, 'queue_time': 0.316429641, 'total_time': 0.053218724}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fcc7-f3e3-75b1-a3fb-4b7fbc0d8a8c-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_7f5b7cb5-926